In [1]:
import sys
from pathlib import Path

import torch
import modelopt.torch.opt as mto
import modelopt.torch.quantization as mtq

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "pyfiles"))
sys.path.insert(0, str(ROOT / "pyfiles" / "qat_modelopt"))

from quantize import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
# 1. Rebuild the model architecture
model = get_model("/home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth", num_classes=100).to(device)

# 2. Restore quantization structure from mostate
mostate = torch.load("/home/pf4636/code/resnet/quantized_resnets/checkpoints/qat/int4_in8b/qat_modelopt_best_mostate.pt", map_location=device)
mto.restore_from_modelopt_state(model, mostate)

# 3. Load weights from pth
ckpt = torch.load("/home/pf4636/code/resnet/quantized_resnets/checkpoints/qat/int4_in8b/qat_modelopt_best.pth", map_location=device)
model.load_state_dict(ckpt["model"])

# 4. Verify quantizers
all_quantizers = [(name, mod) for name, mod in model.named_modules()
                  if isinstance(mod, mtq.nn.TensorQuantizer)]
print(f"Total quantizers: {len(all_quantizers)}")

for name, mod in all_quantizers:
    if "weight" in name:
        print(f"{name}: num_bits={mod.num_bits}, amax={mod.amax}")

Inserted 107 quantizers


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


Total quantizers: 107
conv1.weight_quantizer: num_bits=4, amax=tensor([[[[0.2925]]],


        [[[0.0551]]],


        [[[0.1305]]],


        [[[0.3127]]],


        [[[0.0291]]],


        [[[0.2353]]],


        [[[0.0908]]],


        [[[0.1575]]],


        [[[0.1552]]],


        [[[0.6448]]],


        [[[0.4388]]],


        [[[0.8828]]],


        [[[0.3176]]],


        [[[0.0866]]],


        [[[0.5564]]],


        [[[0.1383]]],


        [[[0.0785]]],


        [[[0.0517]]],


        [[[0.0952]]],


        [[[0.9301]]],


        [[[1.2296]]],


        [[[0.8920]]],


        [[[0.6337]]],


        [[[0.3552]]],


        [[[0.0278]]],


        [[[0.3695]]],


        [[[0.6155]]],


        [[[0.1172]]],


        [[[0.5098]]],


        [[[0.0898]]],


        [[[0.1016]]],


        [[[0.5180]]],


        [[[0.8674]]],


        [[[0.2595]]],


        [[[0.1144]]],


        [[[0.2641]]],


        [[[0.1171]]],


        [[[0.0472]]],


        [[[0.8384]]],


 